# Data Preparation

In this notebook, I load, combine, and clean the fake and real news datasets.

In [2]:
# imports
import pandas as pd
import numpy as np
import re
import string


In [3]:
# Load data
fake_df = pd.read_csv("../data/Fake.csv")
true_df = pd.read_csv("../data/True.csv")


## Explore data

In [4]:
fake_df.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [5]:
true_df.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [6]:
print("True dataset shape:", true_df.shape)
print("Fake dataset shape:", fake_df.shape)

True dataset shape: (21417, 4)
Fake dataset shape: (23481, 4)


In [7]:
print("True dataset columns:", true_df.columns.tolist())
print("Fake dataset columns:", fake_df.columns.tolist())

True dataset columns: ['title', 'text', 'subject', 'date']
Fake dataset columns: ['title', 'text', 'subject', 'date']


In [8]:
true_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21417 entries, 0 to 21416
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    21417 non-null  object
 1   text     21417 non-null  object
 2   subject  21417 non-null  object
 3   date     21417 non-null  object
dtypes: object(4)
memory usage: 669.4+ KB


In [9]:
fake_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23481 entries, 0 to 23480
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    23481 non-null  object
 1   text     23481 non-null  object
 2   subject  23481 non-null  object
 3   date     23481 non-null  object
dtypes: object(4)
memory usage: 733.9+ KB


## Add labels

In [10]:
true_df["label"] = "REAL"
fake_df["label"] = "FAKE"

We manually add labels because each file represents one class.

## Merge the two tables

In [11]:
df = pd.concat([true_df, fake_df], axis=0)

## Shuffle the result table

In [12]:
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## keep only the Useful Columns

In [13]:
df = df[["title", "text", "label"]]

In [14]:
df.head()

,title,text,label
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",FAKE
1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,FAKE
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,FAKE
3,California AG pledges to defend birth control ...,SAN FRANCISCO (Reuters) - California Attorney ...,REAL
4,AZ RANCHERS Living On US-Mexico Border Destroy...,Twisted reasoning is all that comes from Pelos...,FAKE


In [15]:
print("Dataset shape after keeping useful columns:", df.shape)

Dataset shape after keeping useful columns: (44898, 3)


### Checking null values

In [19]:
df.isnull().sum()

title    0
text     0
label    0
dtype: int64

## Drop duplicates lines

In [20]:
print("Number of duplicate rows:", df.duplicated().sum())

Number of duplicate rows: 0


In [21]:
df = df.drop_duplicates()

In [22]:
print("Dataset shape after removing duplicates:", df.shape)
print("Number of duplicate rows after cleaning:", df.duplicated().sum())

Dataset shape after removing duplicates: (39105, 3)
Number of duplicate rows after cleaning: 0


## Checking labels Distribution in the dataset

In [23]:
df["label"].value_counts()

label
REAL    21197
FAKE    17908
Name: count, dtype: int64

## Combing the `text` and `title` columns in one column for more context


In [24]:
df["content"] = df["title"] + " " + df["text"]

In [25]:
df[["title", "text", "content", "label"]].head()

,title,text,content,label
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",BREAKING: GOP Chairman Grassley Has Had Enoug...,FAKE
1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,Failed GOP Candidates Remembered In Hilarious...,FAKE
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,FAKE
3,California AG pledges to defend birth control ...,SAN FRANCISCO (Reuters) - California Attorney ...,California AG pledges to defend birth control ...,REAL
4,AZ RANCHERS Living On US-Mexico Border Destroy...,Twisted reasoning is all that comes from Pelos...,AZ RANCHERS Living On US-Mexico Border Destroy...,FAKE


### Create the cleaning function

In [26]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"<.*?>+", "", text)
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"\w*\d\w*", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [30]:
## Apply the cleaning function
df["content_clean"] = df["content"].apply(clean_text)
df[["content", "content_clean"]].head()

,content,content_clean
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,breaking gop chairman grassley has had enough ...
1,Failed GOP Candidates Remembered In Hilarious...,failed gop candidates remembered in hilarious ...
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,mike pence’s new dc neighbors are hilariously ...
3,California AG pledges to defend birth control ...,california ag pledges to defend birth control ...
4,AZ RANCHERS Living On US-Mexico Border Destroy...,az ranchers living on usmexico border destroy ...


In [31]:
print("Final dataset shape:", df.shape)

Final dataset shape: (39105, 5)


In [32]:
df.head()

,title,text,label,content,content_clean
0,BREAKING: GOP Chairman Grassley Has Had Enoug...,"Donald Trump s White House is in chaos, and th...",FAKE,BREAKING: GOP Chairman Grassley Has Had Enoug...,breaking gop chairman grassley has had enough ...
1,Failed GOP Candidates Remembered In Hilarious...,Now that Donald Trump is the presumptive GOP n...,FAKE,Failed GOP Candidates Remembered In Hilarious...,failed gop candidates remembered in hilarious ...
2,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,Mike Pence is a huge homophobe. He supports ex...,FAKE,Mike Pence’s New DC Neighbors Are HILARIOUSLY...,mike pence’s new dc neighbors are hilariously ...
3,California AG pledges to defend birth control ...,SAN FRANCISCO (Reuters) - California Attorney ...,REAL,California AG pledges to defend birth control ...,california ag pledges to defend birth control ...
4,AZ RANCHERS Living On US-Mexico Border Destroy...,Twisted reasoning is all that comes from Pelos...,FAKE,AZ RANCHERS Living On US-Mexico Border Destroy...,az ranchers living on usmexico border destroy ...


In [33]:
df.to_csv("../data/clean_data.csv", index=False)
print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
